In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.ar_model import AutoReg
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

import os
OUTPUT_DIR = "Code Outputs/EDA Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

# Load and reshape
df = pd.read_excel("Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx")
df["Date"] = pd.to_datetime(df["Date"])
LAKES = sorted(df["Reservoir"].unique())

# wide format: one column per lake, monthly index (NaN before a lake starts)
level = df.pivot(index="Date", columns="Reservoir", values="Level_m").sort_index()
level = level.asfreq("MS")   # enforce a clean monthly frequency

# Summary stats and level overview
summary = []
for lk in LAKES:
    s = level[lk].dropna()
    summary.append({
        "Lake": lk, "N_months": len(s),
        "Start": s.index.min().date(), "End": s.index.max().date(),
        "Mean_m": round(s.mean(), 2), "Std_m": round(s.std(), 3),
        "Min_m": round(s.min(), 2), "Max_m": round(s.max(), 2),
        "Range_m": round(s.max() - s.min(), 2),
    })
summary = pd.DataFrame(summary)
summary.to_csv(out("EDA_01_summary_stats.csv"), index=False)
print("=== 1. SUMMARY STATISTICS ==="); print(summary.to_string(index=False))

fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lk in enumerate(LAKES):
    axes[i].plot(level.index, level[lk], lw=.8)
    axes[i].set_title(lk, fontweight="bold"); axes[i].set_ylabel("Level (m)")
axes[7].set_visible(False)
plt.suptitle("Monthly Water Levels (each lake on its own scale)", fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("EDA_01_levels_overview.svg"), dpi=1000); plt.close()


# STL Decompistion  ->  trend, seasonal, residual, deseasonalised anomaly

trend, seasonal, resid, deseason = {}, {}, {}, {}
fig, axes = plt.subplots(len(LAKES), 1, figsize=(14, 3.2 * len(LAKES)))
for ax, lk in zip(axes, LAKES):
    s = level[lk].dropna()
    res = STL(s, period=12, robust=True).fit()
    trend[lk], seasonal[lk], resid[lk] = res.trend, res.seasonal, res.resid
    deseason[lk] = s - res.seasonal
    ax.plot(s.index, s, color="0.7", lw=.7, label="level")
    ax.plot(res.trend.index, res.trend, color="tab:red", lw=1.5, label="STL trend")
    ax.set_title(f"{lk} - level & STL trend", fontweight="bold"); ax.set_ylabel("m")
    if lk == LAKES[0]: ax.legend(fontsize=8)
plt.suptitle("STL Decomposition: extracted multi-decadal trend per lake", fontweight="bold", y=.999)
plt.tight_layout(); plt.savefig(out("EDA_02_stl_trends.svg"), dpi=1000); plt.close()

# Standardised deseasonalised anomaly
anom = pd.DataFrame({lk: (deseason[lk] - deseason[lk].mean()) / deseason[lk].std()
                     for lk in LAKES})
plt.figure(figsize=(15, 6))
for lk in LAKES:
    plt.plot(anom.index, anom[lk], lw=1.1, alpha=.8, label=lk)
plt.axhline(0, color="k", ls="--", lw=1)
plt.title("Standardised Deseasonalised Anomalies (trend + interannual, seasonal cycle removed)",
          fontweight="bold")
plt.ylabel("z-score"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig(out("EDA_02_anomaly_timeline.svg"), dpi=1000); plt.close()

# Stationarity tests (ADF: H0=unit root/non-stationary; KPSS: H0=stationary)
#  Tested on three versions to justify the differencing order d for ARIMA.

def stationarity(s):
    s = s.dropna()
    adf_p  = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    return adf_p, kpss_p

rows = []
for lk in LAKES:
    for name, s in [("level", level[lk]),
                    ("deseasonalised", deseason[lk]),
                    ("deseason_diff1", deseason[lk].diff())]:
        a, k = stationarity(s)
        rows.append({"Lake": lk, "Series": name,
                     "ADF_p": round(a, 4), "KPSS_p": round(k, 4),
                     "Stationary": "yes" if (a < 0.05 and k > 0.05) else "no"})
stat = pd.DataFrame(rows)
stat.to_csv(out("EDA_03_stationarity.csv"), index=False)
print("\n=== 3. STATIONARITY (sample) ===")
print(stat[stat.Series != "level"].to_string(index=False))

# Seasonality: monthly climatology + seasonal boxplots

# climatology = average seasonal departure from trend, by calendar month
clim = pd.DataFrame({lk: (level[lk] - trend[lk]).groupby(level.index.month).mean()
                     for lk in LAKES})
clim.index.name = "Month"
clim.to_csv(out("EDA_04_monthly_climatology.csv"))
peak = pd.DataFrame({"Peak_month": clim.idxmax(), "Trough_month": clim.idxmin()})
peak.to_csv(out("EDA_04_season_peak_trough.csv"))

plt.figure(figsize=(11, 6))
for lk in LAKES:
    plt.plot(range(1, 13), clim[lk], marker="o", label=lk)
plt.axhline(0, color="k", ls="--", lw=.8)
plt.xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.title("Mean Seasonal Cycle (level minus trend), by calendar month", fontweight="bold")
plt.ylabel("Seasonal departure (m)"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig(out("EDA_04_climatology.svg"), dpi=1000); plt.close()

# seasonal boxplots of the detrended signal
detr = pd.DataFrame({lk: level[lk] - trend[lk] for lk in LAKES})
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lk in enumerate(LAKES):
    d = detr[lk].dropna()
    sns.boxplot(x=d.index.month, y=d.values, ax=axes[i], color="skyblue")
    axes[i].set_title(lk, fontweight="bold"); axes[i].set_xlabel(""); axes[i].set_ylabel("detrended (m)")
axes[7].set_visible(False)
plt.suptitle("Seasonal Distribution by Month (detrended levels)", fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("EDA_04_seasonal_boxplots.svg"), dpi=1000); plt.close()

# ACF / PACF on the stationary series 
# These plots motivate candidate (p, q) orders for the ARIMA step.

fig, axes = plt.subplots(len(LAKES), 2, figsize=(13, 3 * len(LAKES)))
for i, lk in enumerate(LAKES):
    s = deseason[lk].diff().dropna()
    nl = 36
    a = acf(s, nlags=nl); p = pacf(s, nlags=nl)
    ci = 1.96 / np.sqrt(len(s))
    axes[i,0].stem(range(len(a)), a); axes[i,0].axhline(ci, color="r", ls="--"); axes[i,0].axhline(-ci, color="r", ls="--")
    axes[i,0].set_title(f"{lk} ACF", fontsize=10)
    axes[i,1].stem(range(len(p)), p); axes[i,1].axhline(ci, color="r", ls="--"); axes[i,1].axhline(-ci, color="r", ls="--")
    axes[i,1].set_title(f"{lk} PACF", fontsize=10)
plt.suptitle("ACF / PACF of differenced deseasonalised series", fontweight="bold", y=.999)
plt.tight_layout(); plt.savefig(out("EDA_05_acf_pacf.svg"), dpi=1000); plt.close()

# Cross-lake dependence, raw level vs STL residual
#    Left = correlation of raw levels (contaminated by shared trend + season).
#    Right = correlation of residuals (genuine co-variability). The drop between them quantifies how much of the apparent dependence was common trend/season.

resid_df = pd.DataFrame(resid)
corr_raw   = level.corr()                       # pairwise, on overlapping months
corr_resid = resid_df.corr()
corr_raw.to_csv(out("EDA_06_corr_raw.csv")); corr_resid.to_csv(out("EDA_06_corr_residual.csv"))
(corr_raw - corr_resid).round(2).to_csv(out("EDA_06_corr_inflation.csv"))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, M, t in [(axes[0], corr_raw, "RAW levels (trend + season included)"),
                 (axes[1], corr_resid, "STL RESIDUALS (genuine dependence)")]:
    sns.heatmap(M, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                linewidths=.5, ax=ax)
    ax.set_title(t, fontweight="bold")
plt.suptitle("Cross-Lake Correlation: raw vs deseasonalised-detrended", fontweight="bold", y=1.0)
plt.tight_layout(); plt.savefig(out("EDA_06_corr_compare.svg"), dpi=1000); plt.close()
print("\n=== 6. CORRELATION DROP (raw - residual), mean off-diagonal ===")
od = (corr_raw - corr_resid).where(~np.eye(len(LAKES), dtype=bool))
print(f"  mean inflation removed: {od.stack().mean():.2f}")

# Hieracrhical clustering, from residual correlation (distance = 1 - corr)

dist = 1 - corr_resid.values
np.fill_diagonal(dist, 0); dist = (dist + dist.T) / 2
Z = linkage(squareform(dist, checks=False), method="average")
plt.figure(figsize=(10, 5))
dendrogram(Z, labels=corr_resid.columns.tolist(), leaf_rotation=45)
plt.title("Hierarchical Clustering of Lakes (1 - residual correlation)", fontweight="bold")
plt.ylabel("distance"); plt.tight_layout()
plt.savefig(out("EDA_07_dendrogram.svg"), dpi=1000); plt.close()

# PCA / EOF of the residual matrix over the common overlapping period
# PC1 typically captures the shared regional climate mode (e.g. IOD/ENSO).

common = resid_df.dropna()                       # months present for ALL lakes
X = (common - common.mean()) / common.std()      # standardise each lake
U, S, Vt = np.linalg.svd(X.values, full_matrices=False)
var_explained = (S ** 2) / (S ** 2).sum()
pcs = pd.DataFrame(U * S, index=common.index,
                   columns=[f"PC{i+1}" for i in range(len(S))])
loadings = pd.DataFrame(Vt.T, index=common.columns,
                        columns=[f"PC{i+1}" for i in range(len(S))])
pd.DataFrame({"PC": [f"PC{i+1}" for i in range(len(S))],
              "Var_explained": var_explained.round(3),
              "Cumulative": var_explained.cumsum().round(3)}).to_csv(
              out("EDA_08_pca_variance.csv"), index=False)
loadings.round(3).to_csv(out("EDA_08_pca_loadings.csv"))

fig, ax = plt.subplots(1, 2, figsize=(17, 5))
ax[0].bar(range(1, len(S)+1), var_explained); ax[0].set_xlabel("PC"); ax[0].set_ylabel("variance explained")
ax[0].set_title(f"Scree (PC1 = {var_explained[0]*100:.0f}% of shared variance)", fontweight="bold")
ax[1].plot(pcs.index, pcs["PC1"], color="tab:purple")
ax[1].axhline(0, color="k", lw=.8); ax[1].set_title("PC1 time series (regional mode)", fontweight="bold")
plt.tight_layout(); plt.savefig(out("EDA_08_pca.svg"), dpi=1000); plt.close()
print("\n=== 8. PCA: variance explained ===")
print("  PC1 = {:.1f}%, PC1+PC2 = {:.1f}%".format(var_explained[0]*100,
                                                  var_explained[:2].sum()*100))

# Lead-lag dependence, prewhitened, bidirectional ccf

def ar_filter(series, phi):
    p = len(phi); v = series.values; out = np.full(len(v), np.nan)
    for t in range(p, len(v)):
        window = v[t-p:t][::-1]          # v[t-1], v[t-2], ..., v[t-p]
        out[t] = v[t] - np.dot(phi, window)
    return pd.Series(out, index=series.index)

def prewhiten_ccf(x, y, max_lag=24, maxp=12):
    """Box-Jenkins prewhitening on first differences; returns lags, ccf, ci."""
    dx, dy = x.diff().dropna(), y.diff().dropna()
    idx = dx.index.intersection(dy.index); dx, dy = dx.loc[idx], dy.loc[idx]
    # choose AR order on the driver dx by AIC
    best_p, best_aic = 1, np.inf
    for p in range(1, maxp + 1):
        try:
            aic = AutoReg(dx, lags=p, trend="n", old_names=False).fit().aic
            if aic < best_aic: best_aic, best_p = aic, p
        except Exception:
            pass
    phi = AutoReg(dx, lags=best_p, trend="n", old_names=False).fit().params.values
    ex, ey = ar_filter(dx, phi).dropna(), ar_filter(dy, phi).dropna()
    j = ex.index.intersection(ey.index); ex, ey = ex.loc[j], ey.loc[j]
    ex = (ex - ex.mean()) / ex.std(); ey = (ey - ey.mean()) / ey.std()
    lags = range(-max_lag, max_lag + 1)
    vals = [ex.corr(ey.shift(-k)) for k in lags]      # k>0: x leads y
    return np.array(list(lags)), np.array(vals), 1.96 / np.sqrt(len(ex))

# Comparison: naive vs prewhitened (Victoria vs Tanganyika)
xv, yt = resid_df["Lake Victoria"], resid_df["Lake Tanganyika"]
naive_lags = np.arange(-24, 25)
naive = np.array([xv.corr(yt.shift(-k)) for k in naive_lags])
pl, pc, ci = prewhiten_ccf(level["Lake Victoria"], level["Lake Tanganyika"])
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].stem(naive_lags, naive); ax[0].set_title("NAIVE CCF (misleading: pure decay)", fontweight="bold")
ax[1].stem(pl, pc); ax[1].axhline(ci, color="r", ls="--"); ax[1].axhline(-ci, color="r", ls="--")
ax[1].set_title("PREWHITENED CCF (true lead-lag)", fontweight="bold")
for a in ax: a.axhline(0, color="k", lw=.8); a.set_xlabel("lag (months), >0 = Victoria leads Tanganyika")
plt.tight_layout(); plt.savefig(out("EDA_09_ccf_victoria_tanganyika.svg"), dpi=1000); plt.close()

# All pairs: peak prewhitened correlation and the lag it occurs at
pair_rows = []
for i in range(len(LAKES)):
    for jx in range(i + 1, len(LAKES)):
        a, b = LAKES[i], LAKES[jx]
        try:
            lg, cc, _ = prewhiten_ccf(level[a], level[b])
            k = lg[np.nanargmax(np.abs(cc))]
            pair_rows.append({"Lake_x": a, "Lake_y": b,
                              "peak_lag_m": int(k), "peak_ccf": round(cc[np.nanargmax(np.abs(cc))], 3),
                              "lead": a if k > 0 else (b if k < 0 else "contemporaneous")})
        except Exception as e:
            pair_rows.append({"Lake_x": a, "Lake_y": b, "peak_lag_m": np.nan,
                              "peak_ccf": np.nan, "lead": "failed"})
pd.DataFrame(pair_rows).to_csv(out("EDA_09_leadlag_summary.csv"), index=False)
print("\n=== 9. PREWHITENED LEAD-LAG (peak |ccf| per pair) ===")
print(pd.DataFrame(pair_rows).to_string(index=False))

# Trend tests - Mann-Kendall (non-parametric) + Sen's slope

def mann_kendall(x):
    x = np.asarray(x.dropna()); n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n-1) for j in range(i+1, n))
    # variance with tie correction
    _, counts = np.unique(x, return_counts=True)
    tie = sum(c*(c-1)*(2*c+5) for c in counts)
    var = (n*(n-1)*(2*n+5) - tie) / 18
    z = (s-1)/np.sqrt(var) if s > 0 else ((s+1)/np.sqrt(var) if s < 0 else 0)
    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(z)))
    # Sen's slope (per month)
    slopes = [(x[j]-x[i])/(j-i) for i in range(n-1) for j in range(i+1, n)]
    return z, p, np.median(slopes)

rows = []
for lk in LAKES:
    z, p, sen = mann_kendall(level[lk])
    rows.append({"Lake": lk, "MK_Z": round(z, 2), "p_value": round(p, 4),
                 "trend": "increasing" if (p < 0.05 and z > 0) else
                          ("decreasing" if (p < 0.05 and z < 0) else "none"),
                 "Sen_slope_m_per_year": round(sen * 12, 4)})
pd.DataFrame(rows).to_csv(out("EDA_10_trend_tests.csv"), index=False)
print("\n=== 10. MANN-KENDALL TREND TESTS ===")
print(pd.DataFrame(rows).to_string(index=False))

print("\nEDA complete. Tables: EDA_*.csv  |  Figures: EDA_*.svg")

=== 1. SUMMARY STATISTICS ===
           Lake  N_months      Start        End  Mean_m  Std_m   Min_m   Max_m  Range_m
    Lake Albert       370 1995-06-01 2026-03-01  619.92  1.026  618.50  622.84     4.34
    Lake Edward       371 1995-05-01 2026-03-01  914.04  0.349  913.29  914.88     1.60
      Lake Kivu       371 1995-05-01 2026-03-01 1461.39  0.384 1460.60 1462.39     1.79
    Lake Malawi       403 1992-09-01 2026-03-01  474.16  0.738  472.48  476.36     3.88
Lake Tanganyika       401 1992-10-01 2026-02-01  769.56  0.805  768.18  772.31     4.13
   Lake Turkana       402 1992-10-01 2026-03-01  363.51  2.067  360.28  369.34     9.06
  Lake Victoria       402 1992-09-01 2026-02-01 1135.37  0.638 1133.91 1136.93     3.02


/var/folders/7f/sm9pcqr925bb901gjw5bw17h0000gn/T/ipykernel_18173/1044259706.py:86: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = kpss(s, regression="c", nlags="auto")[1]
/var/folders/7f/sm9pcqr925bb901gjw5bw17h0000gn/T/ipykernel_18173/1044259706.py:86: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = kpss(s, regression="c", nlags="auto")[1]
/var/folders/7f/sm9pcqr925bb901gjw5bw17h0000gn/T/ipykernel_18173/1044259706.py:86: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(s, regression="c", nlags="auto")[1]
/var/folders/7f/sm9pcqr925bb901gjw5bw17h0000gn/T/ipykernel_18173/1044259706.py:86: Interpolati


=== 3. STATIONARITY (sample) ===
           Lake         Series  ADF_p  KPSS_p Stationary
    Lake Albert deseasonalised 0.5812  0.0100         no
    Lake Albert deseason_diff1 0.0000  0.1000        yes
    Lake Edward deseasonalised 0.0039  0.0100         no
    Lake Edward deseason_diff1 0.0000  0.1000        yes
      Lake Kivu deseasonalised 0.0818  0.0100         no
      Lake Kivu deseason_diff1 0.0000  0.1000        yes
    Lake Malawi deseasonalised 0.5144  0.0100         no
    Lake Malawi deseason_diff1 0.0000  0.1000        yes
Lake Tanganyika deseasonalised 0.9737  0.0100         no
Lake Tanganyika deseason_diff1 0.0000  0.0442         no
   Lake Turkana deseasonalised 0.9806  0.0100         no
   Lake Turkana deseason_diff1 0.0001  0.1000        yes
  Lake Victoria deseasonalised 0.6979  0.0100         no
  Lake Victoria deseason_diff1 0.0000  0.1000        yes

=== 6. CORRELATION DROP (raw - residual), mean off-diagonal ===
  mean inflation removed: 0.26

=== 8. PCA: va